In [2]:
import numpy as np
import csv

In [3]:
# ============================================
# 1. LEITURA DO CSV
# ============================================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features relacionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # Convertendo o sexo
            sex = 1 if row["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)

In [4]:
# ============================================
# 2. TRATAMENTO DE DADOS
# ============================================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X

def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))



In [5]:
# ============================================
# 3. FUNÇÕES DO MODELO
# ============================================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost

def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta

In [6]:
# ============================================
# 4. TREINAMENTO
# ============================================

def train(X, y, lr, epochs):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=lr, epochs=epochs)

    return beta, X

In [7]:
# ============================================
# 5. PREDIÇÃO + MATRIZ DE CONFUSÃO + ACURÁCIA
# ============================================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

def confusion_matrix(y_true, y_pred):
    y_t = y_true.flatten()
    y_p = y_pred.flatten()

    VP = np.sum((y_t == 1) & (y_p == 1))
    VN = np.sum((y_t == 0) & (y_p == 0))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))

    return VP, VN, FP, FN

def calculate_metrics(VP, VN, FP, FN):
    acc = (VP + VN) / (VP + VN + FP + FN)
    prec = VP / (VP + FP)
    rec = VP / (VP + FN)
    spec = VN / (VN + FP)
    f1_score = 2 * (prec * rec) / (prec + rec)

    return acc, prec, rec, spec, f1_score

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [8]:
# ============================================
# 6. SPLIT TREINO / TESTE
# ============================================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [41]:
# ============================================
# 7. EXECUÇÃO E EXIBIÇÃO DE MÉTRICAS
# ============================================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train, 0.1, 500)

# Processar teste com MESMAS informações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_pred = predict(X_test, beta)

# Métricas da MATRIZ DE CONFUSÃO
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

# Calcular as métricas
acc, prec, rec, spec, f1_score = calculate_metrics(VP, VN, FP, FN)

# X_append([pclass, sex, age, sibsp, parch, fare])
X_bruno = []
X_bruno.append([3, 1, 21, 0, 0, 200])
X_bruno = np.array(X_bruno)
#X_pedro = fill_missing_age(X_pedro)
#X_pedro = normalize(X_pedro)
X_bruno = add_bias(X_bruno)

y_pred2 = predict(X_bruno, beta)

print("Bruno =", y_pred2)

acc = accuracy(y_test, y_pred)

print(f"\nAcurácia: {acc}")

print("\n"+"="*40)
print("========== MATRIZ DE CONFUSÃO ==========")
print(f"Verdadeiros Positivos (VP): {VP}")
print(f"Verdadeiros Negativos (VN): {VN}")
print(f"Falsos Positivos (FP):      {FP}")
print(f"Falsos Negativos (FN):      {FN}")
print("="*40)

print("\n"+"="*40)
print(f"========= MÉTRICAS DE AVALIAÇÃO ========")
print(f"Acurácia:           {acc:.4f} ({(acc*100):.1f}%)")
print(f"Precisão:           {prec:.4f} ({(prec*100):.1f}%)")
print(f"Revocação:          {rec:.4f} ({(rec*100):.1f}%)")
print(f"Especificidade:     {spec:.4f} ({(spec*100):.1f}%)")
print(f"F1 - Score:         {f1_score:.4f} ({(f1_score*100):.1f}%)")
print("="*40)

Epoch 0 - Cost: 0.6802
Epoch 100 - Cost: 0.4584
Epoch 200 - Cost: 0.4497
Epoch 300 - Cost: 0.4483
Epoch 400 - Cost: 0.4480
Bruno = [[1]]

Acurácia: 0.7366412213740458

========== MATRIZ DE CONFUSÃO ==========
Verdadeiros Positivos (VP): 68
Verdadeiros Negativos (VN): 125
Falsos Positivos (FP):      30
Falsos Negativos (FN):      39

========= MÉTRICAS DE AVALIAÇÃO ========
Acurácia:           0.7366 (73.7%)
Precisão:           0.6939 (69.4%)
Revocação:          0.6355 (63.6%)
Especificidade:     0.8065 (80.6%)
F1 - Score:         0.6634 (66.3%)
